# MLP scene classifier on elevation curves, for any results directory

**Input.** One row per (scene, UE, carrier). Each row is the UE's whole elevation sweep of `LSPS` (default
`scene_dataset.MLP_LSPS`: LoS state and delay spread, `is_los@10 … is_los@80`, `lgDS@10 … lgDS@80`).
For walking-UE sweeps (`walk`), `TRAJ` sets the time layout:
- `"full"`: every snapshot, `is_los@<elev>@t<k>`
- `"summary"`: mean/std over the walk, `is_los~mean@<elev>`
- `"t0"`: the first snapshot only, i.e. a static UE

Context the terminal knows (carrier, polarization pair) is added. A UE is the same terminal at every elevation, because UE positions are
seeded by the scene alone. There are no scene-level aggregates.

**Output.** Softmax over {rural, suburban, urban}.

**Usage.**
- Point `RESULTS_DIR` at any sweep directory under `Results/scene_class/` (`base`, `hard`, `matctl`,
  `pilot`, `base_mixedpol`, `_aborted/...`), set the parameters below, and run all cells.
- Partial runs work: an elevation missing for a scene leaves NaNs, which are imputed and flagged.
- `links.pkl` is built on first use and rebuilt when results are newer.
- Outputs go to `Results/scene_class/analysis/mlp/<tag>/`.

**Kernel:** `Python 3.12 (mlp-venv, CUDA)` (`/path/to/mlp-venv`). It has CUDA torch and no Sionna/TF, so it can't
disturb `sionna-venv`'s CUDA libraries. `MLP_KW["device"] = "all"` spreads CV folds over every visible GPU. Check
`nvidia-smi` first: the GPUs may be shared with other runs.

Details: `docs/scene_classification.md` §6.7.

In [ ]:
RESULTS_DIR = "walk"         # name under Results/scene_class/, or an absolute path
PAIR_DIR    = None           # optional linear-pol companion for pol transfer, e.g. "base_mixedpol"
REBUILD     = "auto"         # "auto" (rebuild if results are newer) | True | False
TAG         = None           # output subfolder; defaults to RESULTS_DIR's relative path (+ /TRAJ for walking UEs)

# Inputs: LSPS at each elevation in ELEVATIONS (and walk snapshot, see TRAJ), plus CONTEXT.
LSPS       = None            # None = sd.MLP_LSPS (is_los, lgDS); base's earlier runs used ["is_los", "K_dB", "lgDS"]
TRAJ       = "full"          # walking UEs: "full" (every snapshot) | "summary" (mean/std over the walk) | "t0" (static)
ELEVATIONS = None            # None = every elevation in the sweep; or e.g. (10, 20, 30, 40)
CONTEXT    = ["frequency", "pol_pair"]   # known to the terminal; [] for LSP curves only

MLP_KW = dict(
    # architecture
    hidden=(128, 128, 64),   # one width per hidden layer; () = softmax regression
    activation="relu",       # relu | gelu | tanh | silu | elu | leaky_relu
    dropout=0.1,
    layer_norm=False,
    # optimisation
    optimizer="adamw",       # adamw | adam | sgd
    lr=1e-3,
    momentum=0.9,            # sgd only
    weight_decay=1e-4,
    batch_size=256,
    class_weight=None,       # None | "balanced"
    # early stopping (validation = held-out scene seeds of the training fold)
    max_epochs=300,
    patience=20,
    min_delta=1e-4,          # val-loss improvement that resets patience
    val_frac=0.15,
    # preprocessing (numeric columns; carrier and pol pair are one-hot)
    impute="median",         # median | mean | zero
    add_indicator=True,      # 0/1 missing flag per column with NaNs (K_dB is NaN at NLoS elevations)
    # misc
    random_state=0,          # torch init, batch order, validation split
    device="all",            # all = CV folds spread over every GPU | auto = first GPU | cuda:N | cpu
    verbose=False,           # print val loss per epoch
)

FOLDS      = 5               # StratifiedGroupKFold by scene seed
CV_SEED    = 0               # fold assignment
CTRL_SEEDS = (1, 2, 3)       # shuffled-label control, one CV per permutation seed
UE_KS      = (1, 2, 5, 10, 20)
UE_DRAWS   = 20              # random UE subsets per (scene, carrier) and k
IMP_TEST_FRAC = 0.3          # held-out scene share for permutation importance
IMP_REPEATS   = 5

RUN = dict(cv=True, control=True, ue_agg=True, importance=True, pol_transfer=True)

In [ ]:
import glob, json, os, sys, time, warnings
from pathlib import Path
_starts = [Path(p).resolve().parent for p in [globals().get("__vsc_ipynb_file__")] if p]
_starts.append(Path.cwd().resolve())
REPO = next(d for s in _starts for d in (s, *s.parents, s / "Ray_Tracing") if (d / "src" / "utils").is_dir())
if str(REPO.parent) not in sys.path:
    sys.path.insert(0, str(REPO.parent))
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from Ray_Tracing.src.utils import scene_dataset as sd
from Ray_Tracing.src.utils import scene_classify as sc

SC_ROOT = REPO / "Results" / "scene_class"


def resolve(d):
    p = Path(d) if os.path.isabs(d) else SC_ROOT / d
    if not glob.glob(str(p / "*" / "scenes_manifest.pkl")):
        raise FileNotFoundError(f"{p} is not a sweep directory (no */scenes_manifest.pkl)")
    try:
        rel = p.resolve().relative_to(SC_ROOT.resolve())
    except ValueError:
        rel = Path(p.name)
    return p, rel


def load_sweep(d, rebuild=REBUILD):
    """links frame for a sweep dir; (re)builds Results/scene_class/dataset/<rel>/ as needed."""
    p, rel = resolve(d)
    ds = SC_ROOT / "dataset" / rel
    lp = ds / "links.pkl"
    newest = max(os.path.getmtime(f) for f in glob.glob(str(p / "*" / "*.pkl")))
    if rebuild is True or not lp.exists() or (rebuild == "auto" and newest > os.path.getmtime(lp)):
        sd.build_dataset(str(p.parent), [p.name], out_dir=str(ds.parent))
    return pd.read_pickle(lp), rel


lsps = list(LSPS or sd.MLP_LSPS)
links, REL = load_sweep(RESULTS_DIR)
WALK = links.traj_t_s.nunique() > 1
curves = sd.elevation_curves(links, lsps, ELEVATIONS, traj=TRAJ)
pair_curves = sd.elevation_curves(load_sweep(PAIR_DIR)[0], lsps, ELEVATIONS, traj=TRAJ) if PAIR_DIR else None
ELEVS = sorted(links.elevation.unique()) if ELEVATIONS is None else sorted(ELEVATIONS)
COLS = [c for c in curves.columns if "@" in c] + list(CONTEXT)   # curve columns, in elevation_curves order
assert set(CONTEXT) <= {"frequency", "pol_pair"}, "CONTEXT: frequency and/or pol_pair"

OUT = SC_ROOT / "analysis" / "mlp" / (TAG or (f"{REL}/{TRAJ}" if WALK else str(REL)))
OUT.mkdir(parents=True, exist_ok=True)
with open(OUT / "config.json", "w") as f:
    json.dump(dict(RESULTS_DIR=str(RESULTS_DIR), PAIR_DIR=PAIR_DIR, LSPS=lsps, TRAJ=TRAJ if WALK else None,
                   ELEVATIONS=ELEVS, CONTEXT=CONTEXT,
                   MLP_KW=MLP_KW, FOLDS=FOLDS, CV_SEED=CV_SEED, CTRL_SEEDS=CTRL_SEEDS, features=COLS),
              f, indent=1, default=str)

import torch
from Ray_Tracing.src.utils.scene_mlp import gpu_devices
print(f"torch {torch.__version__}; GPUs: {[torch.cuda.get_device_name(d) for d in gpu_devices()] or 'none (CPU)'}; "
      f"device={MLP_KW['device']!r}")
print(f"{REL}: {len(curves)} UE curves ({curves.scene_seed.nunique()} scenes) -> {OUT}")
_walk = (f" | walking UEs: {links.traj_t_s.nunique()} snapshots, TRAJ={TRAJ!r}, "
         f"speed {links.ue_speed_mps.min():.2f}-{links.ue_speed_mps.max():.2f} m/s") if WALK else ""
print(f"{len(COLS)} inputs: {lsps} x {len(ELEVS)} elevations {ELEVS} + context {CONTEXT}{_walk}")
print("carriers [GHz]:", sorted(curves.frequency.unique() / 1e9), "| pol pairs:", curves.pol_pair.value_counts().to_dict())
display(curves.groupby("scene_class").agg(scenes=("scene_seed", "nunique"), ue_curves=("ue", "size")))
miss = curves[COLS[:len(COLS) - len(CONTEXT)]].isna().mean()
print("NaN share per LSP (all elevations):", miss.groupby(miss.index.str.split("@").str[0]).mean().round(3).to_dict())

## 1 - Grouped CV (scene-seed folds) and shuffled-label control

In [ ]:
RESULTS = {}
if RUN["cv"]:
    t = time.time()
    MLP = sc.grouped_cv(curves, "mlp", cols=COLS, folds=FOLDS, seed=CV_SEED, **MLP_KW)
    MLP.to_pickle(OUT / "cv_predictions.pkl")
    RESULTS["per_ue_accuracy"] = sc.overall_accuracy(MLP)
    print(f"per-UE accuracy {RESULTS['per_ue_accuracy']:.3f}  ({time.time() - t:.0f} s)")
    acc = sc.accuracy_by(MLP, ["frequency", "label"])
    acc.to_csv(OUT / "accuracy_by_carrier_class.csv", index=False)
    display(acc.pivot(index="label", columns="frequency", values="accuracy").round(3)
            .rename_axis(index="true class (recall)"))

if RUN["control"]:
    ctrl = {}
    for s in CTRL_SEEDS:
        c = sc.grouped_cv(curves, "mlp", cols=COLS, folds=FOLDS, seed=s, shuffle_labels=True, **MLP_KW)
        agree = (c.groupby("scene_seed").label.first() == curves.groupby("scene_seed").scene_class.first()).mean()
        ctrl[s] = dict(accuracy=sc.overall_accuracy(c), perm_label_agreement=agree)
    ctrl = pd.DataFrame(ctrl).T
    ctrl.to_csv(OUT / "shuffled_control.csv")
    RESULTS["control_mean"] = ctrl.accuracy.mean()
    print(f"chance = {1 / curves.scene_class.nunique():.3f}; the control tracks each permutation's agreement")
    display(ctrl.round(3))

In [ ]:
if RUN["cv"]:
    freqs = sorted(MLP.frequency.unique())
    fig, axs = plt.subplots(1, len(freqs), figsize=(4.2 * len(freqs), 3.6), squeeze=False)
    for ax, f in zip(axs[0], freqs):
        cm = sc.confusion(MLP[MLP.frequency == f], labels=[c for c in sc.CLASSES if c in set(MLP.label)])
        cm.to_csv(OUT / f"confusion_{f / 1e9:g}GHz.csv")
        ax.imshow(cm, vmin=0, vmax=1, cmap="Blues")
        ax.set(title=f"{f / 1e9:g} GHz", xticks=range(len(cm)), yticks=range(len(cm)), xticklabels=cm.columns,
               yticklabels=cm.index, xlabel="predicted", ylabel="true")
        for i in range(len(cm)):
            for j in range(len(cm)):
                ax.text(j, i, f"{cm.iat[i, j]:.2f}", ha="center", va="center")
    plt.tight_layout(); fig.savefig(OUT / "confusion.png", dpi=150); plt.show()

## 2 - Accuracy vs UEs aggregated (mean log-posterior of k UE curves per scene and carrier)

In [ ]:
if RUN["ue_agg"] and RUN["cv"]:
    ks = tuple(k for k in UE_KS if k <= curves.groupby(["scene_seed", "frequency"]).size().max())
    ue = sc.ue_aggregated_accuracy(MLP, ks=ks, n_draws=UE_DRAWS, by=("frequency",))
    ue.to_csv(OUT / "accuracy_vs_n_ue.csv", index=False)
    RESULTS.update({f"k{k}_accuracy": a for k, a in ue.groupby("k").accuracy.mean().items()})
    display(ue.pivot(index="k", columns="frequency", values="accuracy").round(3))
    fig, ax = plt.subplots(figsize=(5, 3.6))
    for f, g in ue.groupby("frequency"):
        ax.plot(g.k, g.accuracy, "o-", label=f"{f / 1e9:g} GHz")
    ax.axhline(1 / curves.scene_class.nunique(), color="k", lw=0.8, ls=":")
    ax.set(xscale="log", xticks=ks, xticklabels=ks, xlabel="UEs aggregated", ylabel="scene accuracy", ylim=(0, 1.02))
    ax.legend(fontsize=7); plt.tight_layout(); fig.savefig(OUT / "accuracy_vs_n_ue.png", dpi=150); plt.show()

## 3 - Permutation importance (held-out scenes)

There are two views, both measured as the drop in per-UE accuracy:
- **Per LSP:** one LSP's whole curve is shuffled across UEs.
- **Per elevation:** every LSP at one elevation is shuffled together.

- **Per walk snapshot** (walking UEs, `TRAJ="full"`): every LSP at one time step is shuffled together.

Context columns are their own groups.

In [ ]:
if RUN["importance"]:
    imp = {by: sc.curve_importance(curves, "mlp", cols=COLS, by=by, test_frac=IMP_TEST_FRAC,
                                   n_repeats=IMP_REPEATS, **MLP_KW)
           for by in ("lsp", "elevation", *(("time",) if any("@t" in c for c in COLS) else ()))}
    fig, axs = plt.subplots(1, len(imp), figsize=(5 * len(imp), 3.8))
    for ax, (by, s) in zip(axs, imp.items()):
        s.to_csv(OUT / f"importance_by_{by}.csv")
        if by in ("elevation", "time"):   # axis order, context last
            el = [k for k in s.index if k not in CONTEXT]
            s = s.reindex(sorted(el, key=lambda k: float(k.lstrip("t"))) + [k for k in s.index if k in CONTEXT])
        s.plot.barh(ax=ax, title=f"accuracy drop, permuted per {by}")
        ax.invert_yaxis()
    plt.tight_layout(); fig.savefig(OUT / "importance.png", dpi=150); plt.show()

## 4 - Polarization transfer (CP ↔ linear, held-out scenes)

The data source depends on the inputs:
- **`PAIR_DIR` set:** CP curves come from `RESULTS_DIR` and linear curves (V/V, H/H) from `PAIR_DIR`.
  This is the case for `base` + `base_mixedpol`, which share scene seeds.
- **No `PAIR_DIR`, mixed-pol sweep:** both families come from the one sweep, with disjoint scenes
  per pair.
- **Single-pol sweep without `PAIR_DIR`:** the test is skipped.

`pol_pair` is dropped from the inputs here. CP pairs read out of a dual-pol solve under-count diffuse
power by ~1.3× (doc §2.6). Linear pairs are exact.

In [ ]:
has = lambda df, pairs: df is not None and df.pol_pair.isin(pairs).any()
if RUN["pol_transfer"] and has(curves, sc.CP_PAIRS) and (has(pair_curves, sc.LINEAR_PAIRS) or has(curves, sc.LINEAR_PAIRS)):
    PT = sc.pol_transfer(curves, pair_curves if has(pair_curves, sc.LINEAR_PAIRS) else None, cols=COLS,
                         folds=FOLDS, seed=CV_SEED, **MLP_KW)
    PT.to_pickle(OUT / "pol_transfer_predictions.pkl")
    sc.accuracy_by(PT, ["train_pol", "test_pol", "frequency"]).to_csv(OUT / "pol_transfer.csv", index=False)
    pt = sc.accuracy_by(PT, ["train_pol", "test_pol"]).set_index(["train_pol", "test_pol"])
    for ab, g in PT.groupby(["train_pol", "test_pol"]):
        for k, a in sc.ue_aggregated_accuracy(g, ks=(5, 20), n_draws=UE_DRAWS, by=()).set_index("k").accuracy.items():
            pt.loc[ab, f"k = {k}"] = a
    RESULTS.update({f"pol_{a}->{b}": v for (a, b), v in pt.accuracy.items()})
    display(pt.round(3))
else:
    print("pol transfer skipped: need CP and linear rows (set PAIR_DIR for a single-pol sweep)")

In [ ]:
summary = pd.Series(RESULTS, name=str(REL))
summary.to_csv(OUT / "summary.csv")
display(summary.round(3).to_frame())